# Test: Agent & MCP Data Synthesis Flows

Exercises the sample pipelines in this folder:

1. **MCP server connectivity** — list tools + call one tool
2. **Offline flow** — scripted policy + live MCP results
3. **Teacher flow** — `MCPAgentBlock` + `.env` teacher model
4. **Inspect** — summarize saved JSONL trajectories

**Prerequisites**

- Use the local `.venv` (Python 3.12) as the notebook kernel
- MCP server on `http://127.0.0.1:8008/mcp` (started below if needed)
- For teacher flow: `.env` with `TEACHER_MODEL_URL`, `TEACHER_MODEL_NAME`, `TEACHER_API_KEY`

## 0. Setup

In [1]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import time
import uuid
from pathlib import Path
from typing import Any

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

nest_asyncio.apply()

SAMPLE_DIR = Path.cwd().resolve()
for candidate in [SAMPLE_DIR, *SAMPLE_DIR.parents]:
    if (candidate / "mcp_server.py").exists():
        SAMPLE_DIR = candidate
        break

os.chdir(SAMPLE_DIR)
sys.path.insert(0, str(SAMPLE_DIR))
load_dotenv(SAMPLE_DIR / ".env")

MCP_SERVER_URL = os.environ.get("MCP_SERVER_URL", "http://127.0.0.1:8008/mcp")
MCP_HOST, MCP_PORT = "127.0.0.1", 8008

MODEL = os.environ.get("MODEL") or os.environ.get("TEACHER_MODEL_NAME") or ""
API_BASE = os.environ.get("API_BASE") or os.environ.get("TEACHER_MODEL_URL")
API_KEY = (
    os.environ.get("OPENAI_API_KEY")
    or os.environ.get("API_KEY")
    or os.environ.get("TEACHER_API_KEY")
)

print(f"SAMPLE_DIR     : {SAMPLE_DIR}")
print(f"MCP_SERVER_URL : {MCP_SERVER_URL}")
print(f"Teacher model  : {MODEL or '(not set)'}")
print(f"API base       : {API_BASE or '(not set)'}")
print(f"API key set    : {bool(API_KEY)}")

SAMPLE_DIR     : /Users/sudash/ai-projects/boomi/docs/samples/mcp-agent-synthesis
MCP_SERVER_URL : http://127.0.0.1:8008/mcp
Teacher model  : openai/llama-scout-17b
API base       : https://maas-rhdp.apps.maas.redhatworkshops.io/v1
API key set    : True


## 1. Ensure MCP server is running

Starts `mcp_server.py` in the background if port 8008 is free.

In [2]:
def port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


_mcp_proc = globals().get("_mcp_proc")

if port_open(MCP_HOST, MCP_PORT):
    print(f"MCP already listening on {MCP_HOST}:{MCP_PORT}")
else:
    py = SAMPLE_DIR / ".venv" / "bin" / "python"
    if not py.exists():
        py = Path(sys.executable)
    _mcp_proc = subprocess.Popen(
        [str(py), str(SAMPLE_DIR / "mcp_server.py")],
        cwd=str(SAMPLE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(40):
        if port_open(MCP_HOST, MCP_PORT):
            break
        time.sleep(0.25)
    else:
        out = _mcp_proc.stdout.read() if _mcp_proc.stdout else ""
        raise RuntimeError(f"MCP server failed to start.\n{out}")
    print(f"Started MCP server pid={_mcp_proc.pid} on {MCP_HOST}:{MCP_PORT}")

MCP already listening on 127.0.0.1:8008


## 2. Flow A — MCP connectivity smoke test

List tools and execute `list_orders` / `get_order`.

In [3]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def tool_result_text(result: Any) -> str:
    parts: list[str] = []
    for block in getattr(result, "content", []) or []:
        text = getattr(block, "text", None)
        parts.append(text if text is not None else str(block))
    if not parts and getattr(result, "structuredContent", None) is not None:
        return json.dumps(result.structuredContent)
    return "\n".join(parts) if parts else json.dumps({"ok": True})


async def mcp_smoke_test(url: str = MCP_SERVER_URL) -> dict:
    async with streamablehttp_client(url) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            tool_names = [t.name for t in tools.tools]
            orders = await session.call_tool("list_orders", {"status": "paid"})
            order = await session.call_tool("get_order", {"order_id": "ORD-1001"})
            return {
                "tools": tool_names,
                "list_orders": tool_result_text(orders),
                "get_order": tool_result_text(order),
            }


smoke = await mcp_smoke_test()
print("Tools:", smoke["tools"])
print("\nlist_orders(status=paid):")
print(smoke["list_orders"][:500])
print("\nget_order(ORD-1001):")
print(smoke["get_order"][:500])

assert set(smoke["tools"]) >= {
    "list_orders",
    "get_order",
    "check_inventory",
    "create_shipment",
}
print("\n✅ MCP smoke test passed")

Tools: ['list_orders', 'get_order', 'check_inventory', 'create_shipment']

list_orders(status=paid):
[{"id":"ORD-1002","customer":"Northwind","sku":"SKU-MON-27","qty":5,"status":"paid","ship_to":"Seattle, WA"}]

get_order(ORD-1001):
{"id":"ORD-1001","customer":"Acme Retail","sku":"SKU-LAP-14","qty":2,"status":"shipped","ship_to":"Austin, TX"}

✅ MCP smoke test passed


## 3. Flow B — Offline synthesis (no teacher LLM)

Runs scripted multi-tool plans against the live MCP server and writes JSONL.

In [4]:
OFFLINE_OUT = SAMPLE_DIR / "tool_calling_dataset_offline.jsonl"

SYSTEM_PROMPT_OFFLINE = (
    "You are an order-operations agent. Use the available MCP tools to look up "
    "orders, check inventory, and create shipments. Prefer tools over guessing. "
    "When finished, give a concise final answer with IDs and quantities."
)

PLANS: list[dict[str, Any]] = [
    {
        "question": "List all paid orders and tell me which ones we can fulfill from current stock.",
        "calls": [
            ("list_orders", {"status": "paid"}),
            ("check_inventory", {"sku": "SKU-LAP-14"}),
            ("check_inventory", {"sku": "SKU-MON-27"}),
        ],
        "answer": (
            "Paid orders: ORD-1001 (SKU-LAP-14 x2) and ORD-1002 (SKU-MON-27 x5). "
            "Both can be fulfilled from current on-hand inventory."
        ),
    },
    {
        "question": (
            "Look up order ORD-1001, check inventory for its SKU, "
            "and create a UPS shipment if possible."
        ),
        "calls": [
            ("get_order", {"order_id": "ORD-1001"}),
            ("check_inventory", {"sku": "SKU-LAP-14"}),
            ("create_shipment", {"order_id": "ORD-1001", "carrier": "UPS"}),
        ],
        "answer": "ORD-1001 shipped via UPS after confirming inventory for SKU-LAP-14.",
    },
    {
        "question": (
            "Can we ship ORD-1002 today? Check inventory and create a FedEx "
            "shipment if stock allows."
        ),
        "calls": [
            ("get_order", {"order_id": "ORD-1002"}),
            ("check_inventory", {"sku": "SKU-MON-27"}),
            ("create_shipment", {"order_id": "ORD-1002", "carrier": "FedEx"}),
        ],
        "answer": "ORD-1002 can ship today; FedEx shipment created for SKU-MON-27 x5.",
    },
    {
        "question": "Why can't we ship ORD-1003 yet? Inspect the order and inventory.",
        "calls": [
            ("get_order", {"order_id": "ORD-1003"}),
            ("check_inventory", {"sku": "SKU-KB-MECH"}),
        ],
        "answer": (
            "ORD-1003 is still pending payment. Inventory for SKU-KB-MECH is available, "
            "but only paid orders can be shipped."
        ),
    },
    {
        "question": "Summarize on-hand inventory for every SKU referenced by paid orders.",
        "calls": [
            ("list_orders", {"status": "paid"}),
            ("check_inventory", {"sku": "SKU-LAP-14"}),
            ("check_inventory", {"sku": "SKU-MON-27"}),
        ],
        "answer": "Paid-order SKUs inventory summarized from live MCP tool results.",
    },
]


async def run_plan(session: ClientSession, plan: dict[str, Any]) -> dict[str, Any]:
    messages: list[dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT_OFFLINE},
        {"role": "user", "content": plan["question"]},
    ]
    for tool_name, arguments in plan["calls"]:
        call_id = f"call_{uuid.uuid4().hex[:8]}"
        messages.append(
            {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": call_id,
                        "type": "function",
                        "function": {
                            "name": tool_name,
                            "arguments": json.dumps(arguments),
                        },
                    }
                ],
            }
        )
        result = await session.call_tool(tool_name, arguments)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": call_id,
                "name": tool_name,
                "content": tool_result_text(result),
            }
        )

    final = plan["answer"]
    last_tool = next((m for m in reversed(messages) if m.get("role") == "tool"), None)
    if last_tool and "Summarize on-hand" in plan["question"]:
        final = (
            "On-hand inventory for paid-order SKUs (from MCP):\n"
            f"{last_tool['content']}"
        )

    messages.append({"role": "assistant", "content": final})
    return {
        "messages": messages,
        "iterations": len(plan["calls"]) + 1,
        "source_question": plan["question"],
        "mode": "offline_mcp",
    }


async def run_offline_flow() -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    async with streamablehttp_client(MCP_SERVER_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"MCP server : {MCP_SERVER_URL}")
            print(f"Tools      : {[t.name for t in tools.tools]}")
            print(f"Plans      : {len(PLANS)}")
            for plan in PLANS:
                record = await run_plan(session, plan)
                records.append(record)
                n_calls = sum(1 for m in record["messages"] if m.get("tool_calls"))
                print(f"\nQ: {plan['question'][:70]}...")
                print(f"  tool_calls={n_calls}  messages={len(record['messages'])}")
    return records


offline_rows = await run_offline_flow()
with OFFLINE_OUT.open("w", encoding="utf-8") as f:
    for record in offline_rows:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"\nOffline rows: {len(offline_rows)} → {OFFLINE_OUT.name}")
assert len(offline_rows) == len(PLANS)
assert all(any(m.get("tool_calls") for m in r["messages"]) for r in offline_rows)
print("✅ Offline flow passed")

MCP server : http://127.0.0.1:8008/mcp
Tools      : ['list_orders', 'get_order', 'check_inventory', 'create_shipment']
Plans      : 5

Q: List all paid orders and tell me which ones we can fulfill from curren...
  tool_calls=3  messages=9

Q: Look up order ORD-1001, check inventory for its SKU, and create a UPS ...
  tool_calls=3  messages=9

Q: Can we ship ORD-1002 today? Check inventory and create a FedEx shipmen...
  tool_calls=3  messages=9

Q: Why can't we ship ORD-1003 yet? Inspect the order and inventory....
  tool_calls=2  messages=7



Q: Summarize on-hand inventory for every SKU referenced by paid orders....
  tool_calls=3  messages=9



Offline rows: 5 → tool_calling_dataset_offline.jsonl
✅ Offline flow passed


## 4. Flow C — Teacher `MCPAgentBlock` synthesis

Requires `.env` teacher credentials. Skips cleanly if not configured.

In [5]:
from sdg_hub.core.blocks import MCPAgentBlock

SEED_QUERIES = [
    "Look up order ORD-1001, check inventory for its SKU, and create a UPS shipment if possible.",
    "Why can't we ship ORD-1003 yet? Inspect the order and inventory.",
    "Summarize on-hand inventory for every SKU referenced by paid orders.",
]

SYSTEM_PROMPT = (
    "You are an order-operations agent. You MUST call MCP tools to answer — "
    "never invent order IDs, SKUs, quantities, or inventory. "
    "Use list_orders, get_order, check_inventory, and create_shipment as needed. "
    "Call tools first; only after tool results are returned, give a concise final "
    "answer with IDs and quantities. Do not describe planned tool calls in prose."
)

teacher_rows: list[dict] = []
TEACHER_OUT = SAMPLE_DIR / "tool_calling_dataset.jsonl"

if not API_KEY or not MODEL:
    print("⚠️  Skipping teacher flow — set TEACHER_* in .env")
else:
    block = MCPAgentBlock(
        block_name="notebook_mcp_agent",
        mcp_server_url=MCP_SERVER_URL,
        model=MODEL,
        api_key=API_KEY,
        api_base=API_BASE,
        max_iterations=8,
        system_prompt=SYSTEM_PROMPT,
        input_cols=["question"],
        output_cols=["agent_trace"],
    )

    dataset = pd.DataFrame({"question": SEED_QUERIES})
    print(f"Teacher: {MODEL}")
    print(f"Queries: {len(dataset)}")
    result = block(dataset)

    with TEACHER_OUT.open("w", encoding="utf-8") as f:
        for _, row in result.iterrows():
            trace = row["agent_trace"]
            if not isinstance(trace, dict) or trace.get("max_iterations_reached"):
                print(f"  skip incomplete: {row['question'][:60]}...")
                continue
            messages = list(trace.get("messages") or [])
            n_tc = sum(
                1 for m in messages if m.get("tool_calls") or m.get("function_call")
            )
            if n_tc == 0:
                print(f"  skip (no tool calls): {row['question'][:60]}...")
                continue

            record = {
                "messages": messages,
                "iterations": trace.get("iterations"),
                "source_question": row["question"],
                "mode": "teacher_mcp",
                "teacher_model": MODEL,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            teacher_rows.append(record)

            tools = []
            for m in messages:
                for tc in m.get("tool_calls") or []:
                    tools.append((tc.get("function") or {}).get("name"))
            print(f"\nQ: {row['question']}")
            print(f"  tools={tools}  iterations={trace.get('iterations')}")

    print(f"\nWrote {len(teacher_rows)} trajectories → {TEACHER_OUT.name}")
    assert len(teacher_rows) >= 1, "expected at least one teacher trajectory with tool calls"
    print("✅ Teacher flow passed")

/Users/sudash/ai-projects/boomi/docs/samples/mcp-agent-synthesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[00:11:00] INFO     Initialized MCPAgentBlock 'notebook_mcp_agent' with model                ]8;id=5601320;file:///Users/sudash/ai-projects/boomi/repos/sdg_hub/src/sdg_hub/core/blocks/mcp/mcp_agent_block.py\mcp_agent_block.py]8;;\:]8;id=5601321;file:///Users/sudash/ai-projects/boomi/repos/sdg_hub/src/sdg_hub/core/blocks/mcp/mcp_agent_block.py#155\155]8;;\
                    'openai/llama-scout-17b' and MCP server 'http://127.0.0.1:8008/mcp'                            

Teacher: openai/llama-scout-17b
Queries: 3


╭────────────────────────────────────────────── notebook_mcp_agent ───────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: MCPAgentBlock                                                                                       │
│ Input Rows: 3                                                                                                   │
│ Input Columns: 1                                                                                                │
│ Column Names: question                                                                                          │
│ Expected Output Columns: agent_trace                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Starting MCP agent generation for 3 samples                              ]8;id=5601329;file:///Users/sudash/ai-projects/boomi/repos/sdg_hub/src/sdg_hub/core/blocks/mcp/mcp_agent_block.py\mcp_agent_block.py]8;;\:]8;id=5601330;file:///Users/sudash/ai-projects/boomi/repos/sdg_hub/src/sdg_hub/core/blocks/mcp/mcp_agent_block.py#193\193]8;;\

[00:11:04] INFO     MCP agent generation completed for 3 samples                             ]8;id=5601338;file:///Users/sudash/ai-projects/boomi/repos/sdg_hub/src/sdg_hub/core/blocks/mcp/mcp_agent_block.py\mcp_agent_block.py]8;;\:]8;id=5601339;file:///Users/sudash/ai-projects/boomi/repos/sdg_hub/src/sdg_hub/core/blocks/mcp/mcp_agent_block.py#205\205]8;;\

╭───────────────────────────────────────── notebook_mcp_agent - Complete ─────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 3 → 3                                                                                                     │
│ Columns: 1 → 2                                                                                                  │
│ 🟢 Added: agent_trace                                                                                           │
│ 📋 Final Columns: agent_trace, question                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Q: Look up order ORD-1001, check inventory for its SKU, and create a UPS shipment if possible.
  tools=['get_order', 'check_inventory', 'create_shipment']  iterations=3
  skip (no tool calls): Why can't we ship ORD-1003 yet? Inspect the order and invent...
  skip (no tool calls): Summarize on-hand inventory for every SKU referenced by paid...

Wrote 1 trajectories → tool_calling_dataset.jsonl
✅ Teacher flow passed


## 5. Inspect generated data

In [6]:
def summarize_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    summary = []
    for i, r in enumerate(rows):
        tools = []
        for m in r.get("messages", []):
            for tc in m.get("tool_calls") or []:
                tools.append((tc.get("function") or {}).get("name"))
        summary.append(
            {
                "idx": i,
                "mode": r.get("mode"),
                "iterations": r.get("iterations"),
                "n_messages": len(r.get("messages", [])),
                "n_tool_calls": len(tools),
                "tools": ", ".join(t for t in tools if t),
                "question": (r.get("source_question") or "")[:80],
            }
        )
    return pd.DataFrame(summary)


files = [
    SAMPLE_DIR / "tool_calling_dataset_offline.jsonl",
    SAMPLE_DIR / "tool_calling_dataset.jsonl",
]

for path in files:
    print(f"\n=== {path.name} ({path.stat().st_size if path.exists() else 0} bytes) ===")
    df = summarize_jsonl(path)
    if df.empty:
        print("(missing or empty)")
    else:
        display(df)
        print(
            f"rows={len(df)}  total_tool_calls={int(df['n_tool_calls'].sum())}  "
            f"total_messages={int(df['n_messages'].sum())}"
        )


=== tool_calling_dataset_offline.jsonl (8558 bytes) ===


,idx,mode,iterations,n_messages,n_tool_calls,tools,question
0,0,offline_mcp,4,9,3,"list_orders, check_inventory, check_inventory",List all paid orders and tell me which ones we...
1,1,offline_mcp,4,9,3,"get_order, check_inventory, create_shipment","Look up order ORD-1001, check inventory for it..."
2,2,offline_mcp,4,9,3,"get_order, check_inventory, create_shipment",Can we ship ORD-1002 today? Check inventory an...
3,3,offline_mcp,3,7,2,"get_order, check_inventory",Why can't we ship ORD-1003 yet? Inspect the or...
4,4,offline_mcp,4,9,3,"list_orders, check_inventory, check_inventory",Summarize on-hand inventory for every SKU refe...


rows=5  total_tool_calls=14  total_messages=43

=== tool_calling_dataset.jsonl (2476 bytes) ===


,idx,mode,iterations,n_messages,n_tool_calls,tools,question
0,0,teacher_mcp,3,8,3,"get_order, check_inventory, create_shipment","Look up order ORD-1001, check inventory for it..."


rows=1  total_tool_calls=3  total_messages=8


## 6. Peek at one trajectory

In [7]:
peek_path = SAMPLE_DIR / "tool_calling_dataset.jsonl"
if not peek_path.exists() or peek_path.stat().st_size == 0:
    peek_path = SAMPLE_DIR / "tool_calling_dataset_offline.jsonl"

rows = [json.loads(l) for l in peek_path.read_text().splitlines() if l.strip()]
assert rows, f"no trajectories found in {peek_path}"
sample = rows[0]
print(f"File: {peek_path.name}")
print(f"Question: {sample.get('source_question')}\n")

for m in sample["messages"]:
    role = m.get("role")
    if m.get("tool_calls"):
        for tc in m["tool_calls"]:
            fn = tc.get("function") or {}
            print(f"{role}: tool_call {fn.get('name')}({fn.get('arguments')})")
    elif role == "tool":
        print(f"tool: {str(m.get('content'))[:160]}")
    else:
        content = str(m.get("content") or "").replace("\n", " ")
        print(f"{role}: {content[:160]}")

print("\n✅ Peek complete")

File: tool_calling_dataset.jsonl
Question: Look up order ORD-1001, check inventory for its SKU, and create a UPS shipment if possible.

system: You are an order-operations agent. You MUST call MCP tools to answer — never invent order IDs, SKUs, quantities, or inventory. Use list_orders, get_order, check
user: Look up order ORD-1001, check inventory for its SKU, and create a UPS shipment if possible.
assistant: tool_call get_order({"order_id": "ORD-1001"})
assistant: tool_call check_inventory({"sku": "SKU-LAP-14"})
tool: [{"type": "text", "text": "{\"id\":\"ORD-1001\",\"customer\":\"Acme Retail\",\"sku\":\"SKU-LAP-14\",\"qty\":2,\"status\":\"shipped\",\"ship_to\":\"Austin, TX\"}
tool: [{"type": "text", "text": "{\"sku\":\"SKU-LAP-14\",\"name\":\"14in Ultrabook\",\"on_hand\":10,\"warehouse\":\"DFW\"}"}]
assistant: tool_call create_shipment({"order_id": "ORD-1001", "carrier": "UPS"})
tool: [{"type": "text", "text": "{\"error\":\"Order ORD-1001 is shipped; only paid orders can ship\"}"}]
a

## Done

| Flow | Mechanism | Output |
| --- | --- | --- |
| MCP smoke | live `ClientSession` | — |
| Offline | scripted plans + MCP tools | `tool_calling_dataset_offline.jsonl` |
| Teacher | `MCPAgentBlock` | `tool_calling_dataset.jsonl` |